# ClusterGCN pri konstantnoj pokrivenosti grafa

Postojeci Korak 2 drzi broj klastera po *batch*-u na 20 za sve vrednosti K, pa sa porastom
K opada i pokrivenost grafa po koraku, sa 20 odsto pri K = 100 na jedan odsto pri K = 2000.
Zato se u izmerenom padu tacnosti mesaju dva uticaja: sitnije particionisanje i manja
pokrivenost. Ovo merenje ih razdvaja tako sto pokrivenost drzi na 20 odsto za sve K, pa se
broj klastera po *batch*-u skalira sa K.

| K | klastera po batch-u | pokrivenost | koraka po epohi |
|---|---|---|---|
| 100 | 20 | 20 % | 5 |
| 500 | 100 | 20 % | 5 |
| 1000 | 200 | 20 % | 5 |
| 2000 | 400 | 20 % | 5 |

Konfiguracija K = 100 je identicna postojecoj, pa sluzi kao kontrola. Ako ne reprodukuje
postojeci rezultat, postavka nije ispravna i ostalim vrednostima se ne moze verovati.

Pored tacnosti se belezi i broj cvorova i potega po *batch*-u. Pri jednakoj pokrivenosti
cvorova broj zadrzanih potega pokazuje koliko strukture sitnije particionisanje odbacuje,
sto je pretpostavljeni mehanizam.

Protokol je u svemu ostalom identican postojecem Koraku 2: L = 3, 5.000 gradijentnih koraka,
skrivena dimenzija 256, koeficijent ucenja 0,01, ocena na svakih 250 koraka.

Procena trajanja je oko sest minuta po seedu, dakle oko sat vremena za svih deset.

In [ ]:
SEEDOVI = [42, 123, 7, 1, 2, 3, 4, 5, 6, 8]
POKRIVENOST = 0.20

In [ ]:
import os, sys, time, json, gc, csv, random, shutil, logging
from pathlib import Path
from datetime import datetime

import torch
import torch.nn.functional as F
import numpy as np

from torch_geometric.nn import SAGEConv
from torch_geometric.loader import ClusterData, ClusterLoader
from torch_geometric.utils import to_undirected, degree
from ogb.nodeproppred import PygNodePropPredDataset, Evaluator
import torch_geometric.data

torch.serialization.add_safe_globals([
    torch_geometric.data.data.DataEdgeAttr,
    torch_geometric.data.data.DataTensorAttr,
    torch_geometric.data.storage.GlobalStorage,
])

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
device = torch.device('cuda')

logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s',
                    handlers=[logging.StreamHandler(sys.stdout)], force=True)
log = logging.getLogger(__name__)

cc = torch.cuda.get_device_capability(0)
log.info(f'kartica {torch.cuda.get_device_name(0)}, compute capability {cc[0]}.{cc[1]}')
log.info(f'dodeljena jezgra se ne ocitavaju iz procesa, nego iz konfiguracije instance')


def postavi_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    torch.cuda.manual_seed_all(s)


def direktorijum(seed):
    p = Path(f'rezultati_pokrivenost/run_{seed}')
    p.mkdir(parents=True, exist_ok=True)
    return p


def upisi_red(seed, fajl, red):
    p = direktorijum(seed) / fajl
    novi = not p.exists()
    with open(p, 'a', newline='') as f:
        w = csv.DictWriter(f, fieldnames=list(red.keys()))
        if novi:
            w.writeheader()
        w.writerow(red)


def vec_uradjeno(seed, fajl, kolona):
    p = direktorijum(seed) / fajl
    if not p.exists():
        return set()
    with open(p) as f:
        return {r[kolona] for r in csv.DictReader(f)}

## Podesavanja i model

Prepisano iz `dopuna_a100_sve.ipynb` bez izmene, jer se rezultat poredi sa postojecim Korakom 2.

In [ ]:
SKRIVENI, LR = 256, 0.01
K2_KORAKA, K2_SLOJEVA, K2_OCENA_SVAKIH = 5000, 3, 250
KLASTERI = [100, 500, 1000, 2000]
UZORAKA_ZA_STRUKTURU = 20


class GraphSAGE(torch.nn.Module):
    def __init__(self, ulaz, skriveni, izlaz, slojeva, dropout=0.5):
        super().__init__()
        self.convs = torch.nn.ModuleList()
        self.bns = torch.nn.ModuleList()
        self.slojeva, self.dropout = slojeva, dropout
        self.convs.append(SAGEConv(ulaz, skriveni))
        self.bns.append(torch.nn.BatchNorm1d(skriveni))
        for _ in range(slojeva - 2):
            self.convs.append(SAGEConv(skriveni, skriveni))
            self.bns.append(torch.nn.BatchNorm1d(skriveni))
        self.convs.append(SAGEConv(skriveni, izlaz))

    def forward(self, x, edge_index):
        for i in range(self.slojeva - 1):
            x = self.convs[i](x, edge_index)
            x = self.bns[i](x)
            x = F.relu(x)
            x = F.dropout(x, p=self.dropout, training=self.training)
        return self.convs[-1](x, edge_index)


def trening_korak(model, batch, opt, ceo_podgraf=False):
    batch = batch.to(device, non_blocking=True)
    opt.zero_grad()
    out = model(batch.x, batch.edge_index)
    if ceo_podgraf:
        maska = batch.train_mask
        if maska.sum() == 0:
            return
        gubitak = F.cross_entropy(out[maska], batch.y[maska])
    else:
        maska = batch.train_mask[:batch.batch_size]
        if maska.sum() == 0:
            return
        gubitak = F.cross_entropy(out[:batch.batch_size][maska],
                                  batch.y[:batch.batch_size][maska])
    gubitak.backward()
    opt.step()

## Ucitavanje grafa

In [ ]:
dataset = PygNodePropPredDataset(name='ogbn-arxiv', root='podaci')
data = dataset[0]
data.y = data.y.squeeze()
split_idx = dataset.get_idx_split()
evaluator = Evaluator(name='ogbn-arxiv')

data.edge_index = to_undirected(data.edge_index)
data.train_mask = torch.zeros(data.num_nodes, dtype=torch.bool)
data.train_mask[split_idx['train']] = True
log.info(f'ogbn-arxiv: {data.num_nodes:,} cvorova, {data.num_edges:,} potega')


@torch.no_grad()
def oceni(model):
    model.eval()
    x = data.x.to(device, non_blocking=True)
    ei = data.edge_index.to(device, non_blocking=True)
    out = model(x, ei).argmax(dim=-1, keepdim=True).cpu()
    del x, ei
    torch.cuda.empty_cache()
    return {s: evaluator.eval({'y_true': data.y[split_idx[s]].unsqueeze(1),
                               'y_pred': out[split_idx[s]]})['acc']
            for s in ['train', 'valid', 'test']}

## Merenje

In [ ]:
for SEED in SEEDOVI:
    uradjeno = vec_uradjeno(SEED, 'pokrivenost.csv', 'oznaka')
    for K in KLASTERI:
        batch_klastera = int(round(K * POKRIVENOST))
        oznaka = f'K{K}'
        if oznaka in uradjeno:
            continue
        postavi_seed(SEED)
        log.info('-' * 70)
        log.info(f'seed {SEED} | K={K}, {batch_klastera} klastera po batch-u, '
                 f'pokrivenost {batch_klastera/K*100:.0f}%')

        t0 = time.time()
        cluster_data = ClusterData(data, num_parts=K, recursive=False)
        metis_vreme = time.time() - t0

        loader = ClusterLoader(cluster_data, batch_size=batch_klastera, shuffle=True,
                               num_workers=8, persistent_workers=True)
        model = GraphSAGE(data.num_features, SKRIVENI, dataset.num_classes, K2_SLOJEVA).to(device)
        opt = torch.optim.Adam(model.parameters(), lr=LR)

        naj_val, naj_test, naj_korak, vrh = 0.0, float('nan'), 0, 0.0
        trening_vreme, korak, epoha = 0.0, 0, 0
        cvorova, potega = [], []
        torch.cuda.reset_peak_memory_stats()
        while korak < K2_KORAKA:
            epoha += 1
            model.train()
            for batch in loader:
                if len(cvorova) < UZORAKA_ZA_STRUKTURU:
                    cvorova.append(batch.num_nodes)
                    potega.append(batch.num_edges)
                torch.cuda.synchronize()
                t0 = time.time()
                trening_korak(model, batch, opt, ceo_podgraf=True)
                torch.cuda.synchronize()
                vrh = max(vrh, torch.cuda.max_memory_allocated() / 1e6)
                trening_vreme += time.time() - t0
                korak += 1
                if korak % K2_OCENA_SVAKIH == 0:
                    a = oceni(model)
                    if a['valid'] > naj_val:
                        naj_val, naj_test, naj_korak = a['valid'], a['test'], korak
                    model.train()
                    torch.cuda.reset_peak_memory_stats()
                if korak >= K2_KORAKA:
                    break

        upisi_red(SEED, 'pokrivenost.csv', {
            'oznaka': oznaka, 'klastera': K, 'batch_klastera': batch_klastera,
            'pokrivenost_pct': batch_klastera / K * 100, 'slojeva': K2_SLOJEVA,
            'koraka_po_epohi': len(loader), 'ukupno_koraka': korak, 'epoha': epoha,
            'metis_s': metis_vreme, 'naj_val': naj_val, 'naj_test': naj_test,
            'naj_korak': naj_korak, 'cvorova_po_koraku': float(np.mean(cvorova)),
            'potega_po_koraku': float(np.mean(potega)),
            'vreme_po_koraku_ms': trening_vreme / korak * 1000, 'memorija_mb': vrh})
        log.info(f'seed {SEED} | K={K}: test={naj_test*100:.2f}% (korak {naj_korak}), '
                 f'{np.mean(cvorova):.0f} cvorova i {np.mean(potega):.0f} potega po koraku, '
                 f'{trening_vreme/korak*1000:.1f} ms/korak, {vrh:.0f}MB')

        del model, opt, loader, cluster_data
        gc.collect()
        torch.cuda.empty_cache()

log.info('Merenje zavrseno za sve seedove')

## Kontrola i sazetak

Konfiguracija K = 100 mora da reprodukuje postojeci rezultat, jer je identicna.

In [ ]:
for SEED in SEEDOVI:
    d = direktorijum(SEED)
    json.dump({'seed': SEED, 'datum': datetime.now().isoformat(),
               'kartica': torch.cuda.get_device_name(0),
               'compute_capability': f'{cc[0]}.{cc[1]}',
               'pokrivenost': POKRIVENOST, 'koraka': K2_KORAKA, 'slojeva': K2_SLOJEVA,
               'skriveni': SKRIVENI, 'lr': LR},
              open(d / 'sazetak.json', 'w'), indent=2)

redovi = []
for SEED in SEEDOVI:
    p = direktorijum(SEED) / 'pokrivenost.csv'
    if p.exists():
        for r in csv.DictReader(open(p)):
            redovi.append((SEED, r))

log.info('prosek preko svih zavrsenih seedova')
log.info(f"  {'K':>5s} {'klastera':>9s} {'tacnost':>9s} {'cvorova':>9s} {'potega':>10s} {'memorija':>9s}")
for K in KLASTERI:
    grupa = [r for _, r in redovi if int(r['klastera']) == K]
    if not grupa:
        continue
    t = np.mean([float(r['naj_test']) for r in grupa]) * 100
    c = np.mean([float(r['cvorova_po_koraku']) for r in grupa])
    e = np.mean([float(r['potega_po_koraku']) for r in grupa])
    m = np.mean([float(r['memorija_mb']) for r in grupa])
    log.info(f"  {K:5d} {int(grupa[0]['batch_klastera']):9d} {t:8.2f}% {c:9.0f} {e:10.0f} {m:8.0f}MB")

log.info(f'zavrseno seedova: {len({s for s, _ in redovi})} od {len(SEEDOVI)}')

## Arhiviranje rezultata

In [ ]:
import zipfile

arhiva = shutil.make_archive('rezultati_pokrivenost', 'zip', '.', 'rezultati_pokrivenost')

with zipfile.ZipFile(arhiva) as z:
    imena = sorted(z.namelist())
print(f'{arhiva}  {Path(arhiva).stat().st_size / 1e6:.2f} MB, '
      f'fajlova {len([x for x in imena if not x.endswith("/")])}')

from IPython.display import FileLink
FileLink(arhiva)